# Análise de Dados com SQL

### Conversão e tratamento temporal

In [58]:
df_1 = _dntk.execute_sql(
  '-- Esta query prepara os dados para análise, combinando e extraindo componentes de data.\n-- 1. A CTE \'base\' filtra a base de dados, valida os campos de data e renomeia as colunas.\n-- 2. A CTE \'tratada\' extrai o ano, mês e dia da nova coluna de data, facilitando futuras agregações e análises.\n-- O resultado é uma tabela limpa e estruturada, pronta para análises mais aprofundadas.\n\nWITH base AS (\n    SELECT\n        MAKE_DATE("Year", "Month", "Day") AS data_evento,\n        "City" AS categoria,\n        "AvgTemperature" AS valor\n    FROM city_temperature.csv\n    WHERE Year IS NOT NULL\n    AND Month BETWEEN 1 AND 12\n    AND Day BETWEEN 1 AND 31\n    AND AvgTemperature IS NOT NULL\n) , \ntratada AS (\n    SELECT\n        data_evento,\n        categoria,\n        valor,\n        EXTRACT(YEAR FROM data_evento)  AS ano,\n        EXTRACT(MONTH FROM data_evento) AS mes,\n        EXTRACT(DAY FROM data_evento)   AS dia\n    FROM base\n)\n\nSELECT *\nFROM tratada\nLIMIT 100;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_1

,data_evento,categoria,valor,ano,mes,dia
0,1995-01-01,Algiers,64.2,1995,1,1
1,1995-01-02,Algiers,49.4,1995,1,2
2,1995-01-03,Algiers,48.8,1995,1,3
3,1995-01-04,Algiers,46.4,1995,1,4
4,1995-01-05,Algiers,47.9,1995,1,5
...,...,...,...,...,...,...
95,1995-04-06,Algiers,59.0,1995,4,6
96,1995-04-07,Algiers,54.9,1995,4,7
97,1995-04-08,Algiers,54.2,1995,4,8
98,1995-04-09,Algiers,57.8,1995,4,9


### Cálculo do tempo entre eventos consecutivos (gaps e intervalos médios)

In [61]:
df_2 = _dntk.execute_sql(
  '-- Esta query analisa lacunas nos dados, medindo o intervalo de tempo entre as medições de temperatura para cada cidade.\n-- 1. A CTE \'base\' limpa e padroniza os dados de data.\n-- 2. A CTE \'ordenada\' usa a função LAG para encontrar a data do evento anterior em cada cidade.\n-- 3. A CTE \'com_intervalo\' calcula a diferença de dias entre o evento atual e o anterior.\n-- O resultado final mostra o intervalo médio, a maior lacuna e o menor intervalo de dias entre as medições para cada categoria (cidade). \nWITH base AS (\n    SELECT\n        MAKE_DATE("Year", "Month", "Day") AS data_evento,\n        "City" AS categoria,\n        "AvgTemperature" AS valor\n    FROM city_temperature.csv\n    WHERE Year IS NOT NULL\n    AND Month BETWEEN 1 AND 12\n    AND Day BETWEEN 1 AND 31\n    AND AvgTemperature IS NOT NULL\n),\nordenada AS (\n    SELECT\n        categoria,\n        data_evento,\n        valor,\n        LAG(data_evento) OVER (PARTITION BY categoria ORDER BY data_evento) AS evento_anterior\n    FROM base\n),\ncom_intervalo AS (\n    SELECT\n        categoria,\n        data_evento,\n        valor,\n        evento_anterior,\n        data_evento - evento_anterior AS dias_intervalo\n    FROM ordenada\n)\nSELECT\n    categoria,\n    AVG(dias_intervalo) AS intervalo_medio,\n    MAX(dias_intervalo) AS maior_lacuna,\n    MIN(dias_intervalo) AS menor_intervalo\nFROM com_intervalo\nGROUP BY categoria;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_2

,categoria,intervalo_medio,maior_lacuna,menor_intervalo
0,Addis Ababa,71.462449,654881,0
1,Atlantic City,1.000000,1,1
2,Bangkok,1.000000,1,1
3,Colombo,0.999892,1,0
4,Colorado Springs,1.000000,1,1
...,...,...,...,...
316,Pittsburgh,1.000000,1,1
317,Seoul,0.999892,1,0
318,Sofia,0.999892,1,0
319,Tegucigalpa,0.999892,1,0


### Integração de múltiplas granularidades (semana, mês, trimestre)

In [64]:
df_3 = _dntk.execute_sql(
  '-- Esta query consolida dados diários de temperatura em resumos semanais, mensais e trimestrais.\n-- 1. A CTE \'base\' limpa e padroniza os dados de data.\n-- 2. A CTE \'agregacoes\' agrupa os dados por categoria (cidade) e por períodos de tempo (semana, mês, trimestre).\n-- O resultado é um resumo da temperatura média e da quantidade de registros para cada período.\n\nWITH base AS (\n    SELECT\n        MAKE_DATE("Year", "Month", "Day") AS data_evento,\n        "City" AS categoria,\n        "AvgTemperature" AS valor\n    FROM city_temperature.csv\n    WHERE Year IS NOT NULL\n    AND Month BETWEEN 1 AND 12\n    AND Day BETWEEN 1 AND 31\n    AND AvgTemperature IS NOT NULL\n),\nagregacoes AS (\n    SELECT\n        categoria,\n        DATE_TRUNC(\'week\', data_evento) AS semana,\n        DATE_TRUNC(\'month\', data_evento) AS mes,\n        DATE_TRUNC(\'quarter\', data_evento) AS trimestre,\n        AVG(valor) AS temp_media,\n        COUNT(*) AS qtd_registros\n    FROM base\n    GROUP BY categoria, DATE_TRUNC(\'week\', data_evento),\n             DATE_TRUNC(\'month\', data_evento),\n             DATE_TRUNC(\'quarter\', data_evento)\n)\nSELECT *\nFROM agregacoes\nORDER BY categoria, mes;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_3

,categoria,semana,mes,trimestre,temp_media,qtd_registros
0,Abidjan,1995-01-23,1995-01-01,1995-01-01,80.500000,7
1,Abidjan,1995-01-16,1995-01-01,1995-01-01,79.028571,7
2,Abidjan,1995-01-09,1995-01-01,1995-01-01,76.642857,7
3,Abidjan,1995-01-02,1995-01-01,1995-01-01,82.557143,7
4,Abidjan,1994-12-26,1995-01-01,1995-01-01,82.600000,1
...,...,...,...,...,...,...
487533,Zurich,2020-04-06,2020-04-01,2020-04-01,58.400000,7
487534,Zurich,2020-04-27,2020-04-01,2020-04-01,53.475000,4
487535,Zurich,2020-05-04,2020-05-01,2020-04-01,59.000000,7
487536,Zurich,2020-05-11,2020-05-01,2020-04-01,46.700000,3


### Tendência geral

In [94]:
df_5 = _dntk.execute_sql(
  '-- Esta query calcula a média móvel da temperatura para identificar tendências de curto e médio prazo.\n-- 1. A CTE \'base\' limpa e padroniza os dados de data.\n-- 2. A CTE \'movel\' usa a função de janela AVG com \'ROWS BETWEEN\' para calcular a média móvel de 7 e 30 dias para cada categoria (cidade).\n-- O resultado mostra a temperatura diária junto com as médias móveis.\n\nWITH base AS (\n    SELECT\n        MAKE_DATE("Year", "Month", "Day") AS data_evento,\n        "City" AS categoria,\n        "AvgTemperature" AS valor\n    FROM city_temperature.csv\n    WHERE Year IS NOT NULL\n    AND Month BETWEEN 1 AND 12\n    AND Day BETWEEN 1 AND 31\n    AND AvgTemperature IS NOT NULL\n),\nmovel AS (\n    SELECT\n        categoria,\n        data_evento,\n        valor,\n        AVG(valor) OVER (\n            PARTITION BY categoria\n            ORDER BY data_evento\n            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW\n        ) AS media_movel_7d,\n        AVG(valor) OVER (\n            PARTITION BY categoria\n            ORDER BY data_evento\n            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW\n        ) AS media_movel_30d\n    FROM base\n)\nSELECT *\nFROM movel\nORDER BY categoria, data_evento;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_5

,categoria,data_evento,valor,media_movel_7d,media_movel_30d
0,Abidjan,1995-01-01,82.6,82.600000,82.600000
1,Abidjan,1995-01-02,82.3,82.450000,82.450000
2,Abidjan,1995-01-03,81.0,81.966667,81.966667
3,Abidjan,1995-01-04,83.3,82.300000,82.300000
4,Abidjan,1995-01-05,83.4,82.520000,82.520000
...,...,...,...,...,...
2906314,Zurich,2020-05-09,67.1,57.100000,56.370000
2906315,Zurich,2020-05-10,64.7,59.000000,56.560000
2906316,Zurich,2020-05-11,52.0,57.914286,56.310000
2906317,Zurich,2020-05-12,43.5,56.771429,55.726667


### Variação percentual

In [1]:
df_6 = _dntk.execute_sql(
  '-- Esta query compara as variações de temperatura mensal (MoM) para cada cidade.\n-- 1. A CTE \'base\' limpa e padroniza os dados de data.\n-- 2. A CTE \'mensal\' agrega os dados diários para obter a temperatura média por mês e cidade.\n-- 3. A CTE \'indice\' calcula a variação percentual mês a mês (MoM) e um índice de temperatura base 100 usando funções de janela.\n-- O resultado final mostra o valor mensal da temperatura, a variação percentual e o índice para cada cidade, permitindo analisar as flutuações e tendências de curto prazo.\n\nWITH base AS (\n    SELECT\n        MAKE_DATE("Year", "Month", "Day") AS data_evento,\n        "City" AS categoria,\n        "AvgTemperature" AS valor\n    FROM city_temperature.csv\n    WHERE "Year" IS NOT NULL\n      AND "Month" BETWEEN 1 AND 12\n      AND "Day" BETWEEN 1 AND 31\n      AND "AvgTemperature" IS NOT NULL\n),\nmensal AS (\n    SELECT\n        DATE_TRUNC(\'month\', data_evento) AS mes,\n        categoria,\n        AVG(valor) AS valor\n    FROM base\n    GROUP BY mes, categoria\n),\nindice AS (\n    SELECT\n        mes,\n        categoria,\n        valor,\n        (valor / FIRST_VALUE(valor) OVER (PARTITION BY categoria ORDER BY mes)) * 100 AS indice,\n        (valor - LAG(valor) OVER (PARTITION BY categoria ORDER BY mes)) \n            / LAG(valor) OVER (PARTITION BY categoria ORDER BY mes) * 100 AS variacao_pct\n    FROM mensal\n)\nSELECT *\nFROM indice\nORDER BY categoria, mes;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_6

,mes,categoria,valor,indice,variacao_pct
0,1995-01-01,Abidjan,79.916129,100.000000,NaN
1,1995-02-01,Abidjan,82.614286,103.376235,3.376235
2,1995-03-01,Abidjan,82.545161,103.289739,-0.083671
3,1995-04-01,Abidjan,83.320000,104.259304,0.938685
4,1995-05-01,Abidjan,82.403226,103.112134,-1.100305
...,...,...,...,...,...
93760,2020-01-01,Zurich,36.683871,114.671776,-5.571701
93761,2020-02-01,Zurich,43.165517,134.933048,17.668927
93762,2020-03-01,Zurich,42.645161,133.306443,-1.205490
93763,2020-04-01,Zurich,55.076667,172.166650,29.151034


### Rolling windows

In [187]:
df_7 = _dntk.execute_sql(
  '-- Esta query calcula a média móvel mensal (rolling average) para suavizar flutuações e revelar tendências de temperatura de curto e médio prazo.\n-- 1. A CTE \'base\' limpa e padroniza os dados de data.\n-- 2. A CTE \'mensal\' agrega os dados diários para obter a temperatura média por mês e cidade.\n-- 3. A CTE \'indice\' calcula a média móvel da temperatura, usando a função de janela AVG em janelas de 3 e 5 meses. Ela também calcula o índice e a variação mensal.\n-- O resultado final inclui o valor mensal da temperatura, a variação, e as médias móveis, proporcionando uma visão mais clara das tendências ao longo do tempo.\n\nWITH base AS (\n    SELECT\n        MAKE_DATE("Year", "Month", "Day") AS data_evento,\n        "City" AS categoria,\n        "AvgTemperature" AS valor\n    FROM city_temperature.csv\n    WHERE "Year" IS NOT NULL\n      AND "Month" BETWEEN 1 AND 12\n      AND "Day" BETWEEN 1 AND 31\n      AND "AvgTemperature" IS NOT NULL\n),\nmensal AS (\n    SELECT\n        DATE_TRUNC(\'month\', data_evento) AS mes,\n        categoria,\n        AVG(valor) AS valor\n    FROM base\n    GROUP BY mes, categoria\n),\nindice AS (\n    SELECT\n        mes,\n        categoria,\n        valor,\n        (valor / FIRST_VALUE(valor) OVER (PARTITION BY categoria ORDER BY mes)) * 100 AS indice,\n        (valor - LAG(valor) OVER (PARTITION BY categoria ORDER BY mes)) \n            / LAG(valor) OVER (PARTITION BY categoria ORDER BY mes) * 100 AS variacao_pct,\n        AVG(valor) OVER (\n            PARTITION BY categoria \n            ORDER BY mes \n            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW\n        ) AS rolling_3,\n        AVG(valor) OVER (\n            PARTITION BY categoria \n            ORDER BY mes \n            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW\n        ) AS rolling_5\n    FROM mensal\n)\nSELECT *\nFROM indice\nORDER BY categoria, mes;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_7

,mes,categoria,valor,indice,variacao_pct,rolling_3,rolling_5
0,1995-01-01,Abidjan,79.916129,100.000000,NaN,79.916129,79.916129
1,1995-02-01,Abidjan,82.614286,103.376235,3.376235,81.265207,81.265207
2,1995-03-01,Abidjan,82.545161,103.289739,-0.083671,81.691859,81.691859
3,1995-04-01,Abidjan,83.320000,104.259304,0.938685,82.826482,82.098894
4,1995-05-01,Abidjan,82.403226,103.112134,-1.100305,82.756129,82.159760
...,...,...,...,...,...,...,...
93760,2020-01-01,Zurich,36.683871,114.671776,-5.571701,39.151864,45.913677
93761,2020-02-01,Zurich,43.165517,134.933048,17.668927,39.565925,42.727448
93762,2020-03-01,Zurich,42.645161,133.306443,-1.205490,40.831516,40.653254
93763,2020-04-01,Zurich,55.076667,172.166650,29.151034,46.962448,43.283921


### Valores acumulados

In [4]:
df_8 = _dntk.execute_sql(
  '-- Esta query analisa a variação de temperatura ano a ano (YoY) para cada cidade.\n-- 1. A CTE \'base\' limpa e padroniza os dados de data.\n-- 2. A CTE \'agregacoes_anuais\' consolida os dados diários, calculando a temperatura média, máxima e mínima para cada ano e cidade.\n-- 3. A CTE \'comparacao\' utiliza a função LAG para calcular a variação percentual da temperatura média anual em relação ao ano anterior.\n-- O resultado final mostra a variação ano a ano para cada cidade, permitindo a análise de tendências de longo prazo.\n\nWITH base AS (\n    SELECT\n        MAKE_DATE("Year", "Month", "Day") AS data_evento,\n        "City" AS categoria,\n        "AvgTemperature" AS valor\n    FROM city_temperature.csv\n    WHERE "Year" IS NOT NULL\n      AND "Month" BETWEEN 1 AND 12\n      AND "Day" BETWEEN 1 AND 31\n      AND "AvgTemperature" IS NOT NULL\n),\nmensal AS (\n    SELECT\n        DATE_TRUNC(\'month\', data_evento) AS mes,\n        categoria,\n        AVG(valor) AS valor\n    FROM base\n    GROUP BY mes, categoria\n),\nacumulado AS (\n    SELECT\n        mes,\n        categoria,\n        valor,\n        (valor / FIRST_VALUE(valor) OVER (PARTITION BY categoria ORDER BY mes)) * 100 AS indice,\n        (valor - LAG(valor) OVER (PARTITION BY categoria ORDER BY mes)) \n            / LAG(valor) OVER (PARTITION BY categoria ORDER BY mes) * 100 AS variacao_pct,\n        SUM(valor) OVER (\n            PARTITION BY categoria \n            ORDER BY mes\n            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n        ) AS acumulado\n    FROM mensal\n)\nSELECT *\nFROM acumulado\nORDER BY categoria, mes;\n\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_8

,mes,categoria,valor,indice,variacao_pct,acumulado
0,1995-01-01,Abidjan,79.916129,100.000000,NaN,79.916129
1,1995-02-01,Abidjan,82.614286,103.376235,3.376235,162.530415
2,1995-03-01,Abidjan,82.545161,103.289739,-0.083671,245.075576
3,1995-04-01,Abidjan,83.320000,104.259304,0.938685,328.395576
4,1995-05-01,Abidjan,82.403226,103.112134,-1.100305,410.798802
...,...,...,...,...,...,...
93760,2020-01-01,Zurich,36.683871,114.671776,-5.571701,14754.800568
93761,2020-02-01,Zurich,43.165517,134.933048,17.668927,14797.966086
93762,2020-03-01,Zurich,42.645161,133.306443,-1.205490,14840.611247
93763,2020-04-01,Zurich,55.076667,172.166650,29.151034,14895.687913


### Comparação entre períodos

In [199]:
df_9 = _dntk.execute_sql(
  '-- Esta query compara as variações de temperatura mês a mês (MoM) e ano a ano (YoY) para cada cidade.\n-- 1. A CTE \'base\' limpa e padroniza os dados de data.\n-- 2. A CTE \'mensal\' agrega os dados diários para obter a temperatura média por mês e cidade.\n-- 3. A CTE \'comparacao\' usa funções de janela (LAG) para calcular o índice de temperatura e a variação mensal.\n-- 4. A CTE \'yoy\' faz um auto-join para calcular a variação anual (YoY) de forma precisa, garantindo a comparação com o mesmo mês do ano anterior.\n-- O resultado final mostra a temperatura média mensal, o índice e as variações (MoM e YoY), permitindo uma análise detalhada das tendências de curto e longo prazo.\n\nWITH base AS (\n    SELECT\n        MAKE_DATE(CAST("Year" AS int), CAST("Month" AS int), CAST("Day" AS int)) AS data_evento,\n        "City" AS categoria,\n        "AvgTemperature" AS valor\n    FROM city_temperature.csv\n    WHERE "Year" IS NOT NULL\n    AND "Month" > 0 AND "Month" <= 12\n    AND "Day" > 0 AND "Day" <= 31\n    AND "AvgTemperature" IS NOT NULL\n),\nmensal AS (\n    SELECT\n        DATE_TRUNC(\'month\', data_evento) AS mes,\n        categoria,\n        AVG(valor) AS valor\n    FROM base\n    GROUP BY mes, categoria\n),\ncomparacao AS (\n    SELECT\n        mes,\n        categoria,\n        valor,\n        -- Índice base 100\n        (valor / FIRST_VALUE(valor) OVER (PARTITION BY categoria ORDER BY mes)) * 100 AS indice,\n        -- Variação percentual mês a mês (MoM)\n        (valor - LAG(valor) OVER (PARTITION BY categoria ORDER BY mes)) \n            / LAG(valor) OVER (PARTITION BY categoria ORDER BY mes) * 100 AS variacao_mom\n    FROM mensal\n)\n, yoy AS (\n    SELECT\n        c.mes,\n        c.categoria,\n        c.valor,\n        c.indice,\n        c.variacao_mom,\n        (c.valor - y.valor) / y.valor * 100 AS variacao_yoy\n    FROM comparacao AS c\n    LEFT JOIN comparacao AS y\n        ON c.categoria = y.categoria\n        AND c.mes = y.mes + INTERVAL \'1 year\'\n)\nSELECT *\nFROM yoy\nORDER BY categoria, mes;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_9

,mes,categoria,valor,indice,variacao_mom,variacao_yoy
0,1995-01-01,Abidjan,79.916129,100.000000,NaN,NaN
1,1995-02-01,Abidjan,82.614286,103.376235,3.376235,NaN
2,1995-03-01,Abidjan,82.545161,103.289739,-0.083671,NaN
3,1995-04-01,Abidjan,83.320000,104.259304,0.938685,NaN
4,1995-05-01,Abidjan,82.403226,103.112134,-1.100305,NaN
...,...,...,...,...,...,...
93760,2020-01-01,Zurich,36.683871,114.671776,-5.571701,30.129305
93761,2020-02-01,Zurich,43.165517,134.933048,17.668927,10.528988
93762,2020-03-01,Zurich,42.645161,133.306443,-1.205490,-5.287290
93763,2020-04-01,Zurich,55.076667,172.166650,29.151034,26.438629


In [175]:
df_10 = _dntk.execute_sql(
  'SELECT\n        MAKE_DATE("Year", "Month", "Day") AS data_evento,\n        "City" AS categoria,\n        "AvgTemperature" AS valor\n    FROM city_temperature.csv\n    WHERE "Year" is not null\n    AND "Month" BETWEEN 1 AND 12\n    AND "Day" BETWEEN 1 AND 31\n    AND "AvgTemperature" IS NOT NULL',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_10

,data_evento,categoria,valor
0,1995-01-01,Algiers,64.2
1,1995-01-02,Algiers,49.4
2,1995-01-03,Algiers,48.8
3,1995-01-04,Algiers,46.4
4,1995-01-05,Algiers,47.9
...,...,...,...
2906314,2013-07-27,San Juan Puerto Rico,82.4
2906315,2013-07-28,San Juan Puerto Rico,81.6
2906316,2013-07-29,San Juan Puerto Rico,84.2
2906317,2013-07-30,San Juan Puerto Rico,83.8


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=1d66470e-937e-4aac-9f14-1f15a78e4843' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>